# Intermediary Asset Pricing

The model of [He and Krishnamurthy (2013)](https://doi.org/10.1257/aer.103.2.732), adapted from an example of EconPDEs.jl. Households hold the risky asset only through specialist intermediaries, and their equity stake in an intermediary is capped at a multiple `m` of the intermediary's net worth. When the specialists' wealth share `x` falls, the sector's capacity to bear risk shrinks, the constraint binds and the price of risk rises.

In [ ]:
Pollis.packages("EconPDEs", "Plots")

## From EconPDEs.jl

The code is the package's example, with Greek letters written in Latin (`lambda`, `sigma`, `rho`, `gamma`, `kappa`, `alpha`).

### The model

The unknown is the specialists' wealth-consumption ratio `pS`. Goods-market clearing ties the aggregate price-consumption ratio `p` to it.

In [ ]:
using EconPDEs, Plots

Base.@kwdef struct HeKrishnamurthy
  # See Table 2-Parameters and Targets
  m::Float64 = 4  # intermediation multiplier
  lambda::Float64 = 0.6 # Debt/assets of intermediary sector
  g::Float64 = 0.02 # dividend growth
  sigma::Float64 = 0.09 # dividend volatility
  rho::Float64 = 0.04 # time discount rate
  gamma::Float64 = 2.0 # risk aversion specialist
  l::Float64 = 1.84 # labor income ratio
end

In [ ]:
m = HeKrishnamurthy()

### The grid and the guess

The grid runs over (0, 1) without its endpoints and is raised to the power 1.5, which moves points toward the lower boundary, where specialists are poorly capitalized and the constraint binds.

In [ ]:
xn = 100
stategrid =  (; x = range(0, 1, length = xn+2)[2:(end-1)].^1.5)

In [ ]:
guess = (; pS = ones(length(stategrid[:x])))

### The equation

Inside the function, `m` is the intermediation multiplier, so the model itself is called `model`. Market clearing gives `p` and its derivatives from `pS`. The volatility of the state feeds back through prices, so the upwind direction comes from the drift `mux`: forward at first, and backward if the drift turns out negative.

In [ ]:
function (model::HeKrishnamurthy)(state::NamedTuple, u::NamedTuple)
  # `m` here is the intermediation multiplier (a model parameter), so the model instance is `model`.
  (; m, lambda, g, sigma, rho, gamma, l) = model
  (; pS, pSx_up, pSx_down, pSxx) = u
  (; x) = state
  iter = 0
  pSx = pSx_up
  @label start
  # market clearing for goods gives x / pS + (1 - x) * rho = (1 + l) / p
  # This allows to obtain p (and its derivatives) in terms of pS (and its derivatives)
  p = (1 + l) / (x / pS + (1 - x) * rho)
  px = - (1 + l) / (x / pS + (1 - x) * rho)^2 * (1 / pS - rho - x / pS^2 * pSx)
  pxx = - (1 + l) / (x / pS + (1 - x) * rho)^2 * (- 1 / pS^2 * pSx - 1 / pS^2 * pSx - x / pS^2 * pSxx + 2 * x / pS^3 * pSx^2) + 2 * (1 + l) / (x / pS + (1 - x) * rho)^3 * (1 / pS - rho - x / pS^2 * pSx)^2

  alphaI = 1 / (1 - lambda * (1 - x))
  if x * (1 + m) * alphaI < 1
    alphaI = 1 / (x * (1 + m))
  end
  # we have the usual feeback loop between asset prices and wealth
  # sigmax = x * (alphaI - 1) * (sigma + sigmap)
  sigmax = x * (alphaI - 1) * sigma / (1 -  x * (alphaI - 1) * px / p)
  sigmapS = pSx / pS * sigmax
  sigmap = px / p * sigmax
  sigmaR = sigma + sigmap
  kappa = gamma * (alphaI * sigmaR - sigmapS)
  mux = x * (1 - x) * ((alphaI - (1 - alphaI * x) / (1-x)) * kappa * sigmaR - 1 / pS + rho - (alphaI - (1 - alphaI * x) / (1 - x)) * sigmaR^2 - l / ((1 - x) * p))
  if (iter == 0) && (mux < 0)
    iter += 1
    pSx = pSx_down
    @goto start
  end
  mupS = pSx / pS * mux + 0.5 * pSxx / pS * sigmax^2
  mup = px / p * mux + 0.5 * pxx / p * sigmax^2
  r = 1 / p + g + mup + sigma * sigmap - kappa * sigmaR
  sigmaCS = kappa / gamma
  muCS = (r - rho) / gamma + (1 + 1 / gamma) / (2 * gamma) * kappa^2
  pSt = - (1 / pS + muCS + mupS + sigmaCS * sigmapS - r - kappa * (sigmaCS + sigmapS))
  muR = r + kappa * sigmaR
  sigmaI = alphaI * sigmaR
  (; pSt), (; pS, p, r, kappa, sigmap, muR, sigmaR, alphaI, mux, sigmax, x, px, pxx, sigmaI)
end

### Solving it

In [ ]:
result = pdesolve(m, stategrid, guess)

### The solution

The price of risk is high when specialists are poorly capitalized, the region where the constraint binds and the sector can absorb little risk.

In [ ]:
xs = stategrid[:x]
plot(xs, result.saved.kappa; xlabel = "specialists' wealth share x", ylabel = "price of risk kappa", legend = false)

## Source and Licence

The code under **From EconPDEs.jl** is adapted from `examples/asset_pricing/he_krishnamurthy.jl` of [EconPDEs.jl](https://github.com/matthieugomez/EconPDEs.jl), version 3.1.0, by Matthieu Gomez. Changes made in Pollis: Greek letters written in Latin.

The notice below covers the package's code only, not the parts added in Pollis.

The EconPDEs.jl package is licensed under the MIT "Expat" License:

> Copyright (c) 2015: Matthieu Gomez.
>
> Permission is hereby granted, free of charge, to any person obtaining
> a copy of this software and associated documentation files (the
> "Software"), to deal in the Software without restriction, including
> without limitation the rights to use, copy, modify, merge, publish,
> distribute, sublicense, and/or sell copies of the Software, and to
> permit persons to whom the Software is furnished to do so, subject to
> the following conditions:
>
> The above copyright notice and this permission notice shall be
> included in all copies or substantial portions of the Software.
>
> THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
> EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF
> MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.
> IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY
> CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,
> TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
> SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

## References

He, Z. and Krishnamurthy, A. (2013). Intermediary asset pricing. *American Economic Review* 103(2), 732–770. [doi:10.1257/aer.103.2.732](https://doi.org/10.1257/aer.103.2.732)